# 04 – Evaluate model on test split

This notebook loads a trained `AlphaEarthTomatoModel` checkpoint and evaluates
chip-level accuracy on the `test` split from `chips_index`.

In [ ]:
from pathlib import Path, sys

# Ensure repo root is on sys.path (assumes this notebook lives under repo_root/notebooks/...)
repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root))
print("Repo root added to sys.path:", repo_root)

In [ ]:
import json
import torch
from torch.utils.data import DataLoader

from src.utils.paths import REPO_ROOT, load_paths_config
from src.modeling.alpha_earth_dataset import AlphaEarthChipsDataset, collate_chips
from src.modeling.alpha_earth_model import AlphaEarthTomatoModel


cfg = load_paths_config()
splits_dir = REPO_ROOT / cfg.get("data", {}).get("splits", "data/splits")
index_path = splits_dir / "chips_index.csv"
print("Using index:", index_path)

# Pick latest run whose name contains "baseline_small_epochs_gpu" (adjust if needed)
exp_root = REPO_ROOT / "logs" / "experiments"
run_dirs = sorted(exp_root.glob("*baseline_small_epochs_gpu*"), reverse=True)
assert run_dirs, "No baseline_small_epochs_gpu run found in logs/experiments"
exp_dir = run_dirs[0]
ckpt_path = exp_dir / "model_final.pt"
print("Using checkpoint:", ckpt_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

test_ds = AlphaEarthChipsDataset(index_path=index_path, split="test")
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2, collate_fn=collate_chips)

model = AlphaEarthTomatoModel(in_channels=64, base_channels=32, dropout_p=0.3).to(device)
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state["model_state_dict"])
model.eval()

def accuracy_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).long()
    return (preds == labels.long()).float().mean().item()

test_acc_sum = 0.0
n_batches = 0

with torch.no_grad():
    for batch in test_loader:
        x = batch.image.to(device)
        y = batch.label.to(device).float()
        _, chip_logits = model(x)
        acc = accuracy_from_logits(chip_logits, y)
        test_acc_sum += acc
        n_batches += 1

test_acc = test_acc_sum / max(1, n_batches)
print(f"Test chip accuracy: {test_acc:.3f}")